<a href="https://colab.research.google.com/github/manikantavs01/DeepLearning_Hackaton/blob/main/genai_support_ticket_classification_fewshot_groq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install Pandas

In [ ]:
!pip install --upgrade typing_extensions

In [3]:
#importing libraries
import os
import pandas as pd
import ast
import time
import re

In [4]:
from getpass import getpass

key = getpass('Please enter your together AI API Key here: ')

Please enter your together AI API Key here:  ········


In [5]:
os.environ['TOGETHER_API_KEY'] = key

In [ ]:
#installing groq
#!pip install together

In [43]:
#setting client and model
from together import Together
client=Together()
model="deepseek-ai/DeepSeek-R1-Distill-Llama-70B-free"

In [8]:
# Reding test and train data
test = pd.read_csv('test_genai.csv')
train = pd.read_csv('train_genai.csv')

In [9]:
#chat function
def get_response(prompt, model=model):
    messages = [{"role":"user","content":prompt}]
    client = Together()
    response = client.chat.completions.create(model=model,messages=messages)
    return response.choices[0].message.content

In [10]:
#Few shot examples
few_shot_examples = f''' You are an helpful customer support ticket classification AI assistant. Given a ticket your job is to classfiy it into one of the following categories
  department as one of Technical Support, Customer Service, Billing and Payments, Product Support, IT Support, Returns&Exchanges, Sales and Pre-Sales, Human Resources, Service Outages and Maintenance, General Inquiry
  type as one of Incident, Request, Change, problem
  priority as low, medium, high
  language as the language code for the email's language
  few examples are provided as few shot examples in the following format
  Strictly output only classification without any additional text'''

# Add the 8 labeled training emails as few-shot examples
for _, row in train.iterrows():
    few_shot_examples += f'Ticket: "{row["ticket_body"]}"\n' \
                       f'Department: {row["department"]}\n' \
                       f'Type: {row["type"]}\n' \
                       f'Priority: {row["priority"]}\n' \
                       f'Language: {row["language"]}\n\n'

In [11]:
# Function to classify test emails in batches
def classify_tickets(tickets):
    classified_output = []
    for email in tickets:
      input_prompt = few_shot_examples + f'Ticket: "{email}"\n' \
                                        f'Department:\nType:\nPriority:\nLanguage:'
      #print(input_prompt)
      try:
        response = get_response(input_prompt)
        #print(response)
        pattern = r"Department:\s*(.*?)\s*Type:\s*(.*?)\s*Priority:\s*(.*?)\s*Language:\s*(.*)"
        match = re.search(pattern, response)
        classified_output.append({
              "email": email,
              "department": match.group(1),
              "type": match.group(2),
              "priority": match.group(3),
              "language": match.group(4)
              })
      except Exception as e:
        classified_output.append({
            "email": email,
            "department": "Error",
            "type": "Error",
            "priority": "Error",
            "language": "Error"
              })
      #time.sleep(0.5) # Prevent API rate limit issues
    return(classified_output)

In [65]:
# Run classification on the test emails
test_emails = test["ticket_body"].tolist()
classified_data = classify_tickets(test_emails[401:501])

In [66]:
classified_data=pd.DataFrame(classified_data)

In [67]:
classified_data.head()

,email,department,type,priority,language
0,"Sehr geehrter Kundenservice,\n\nIch hoffe, es ...",Technical Support,Incident,high,de
1,"प्रिय टेक ऑनलाइन स्टोर सहायता,\n\nमुझे आशा है ...",Technical Support,Incident,high,hi
2,"Sehr geehrter Kundenservice,\n\nAuf meiner let...",Billing and Payments,Incident,low,de
3,Estimado servicio de atención al cliente:\n\nL...,Technical Support,Problem,medium,es
4,Chère équipe d'assistance de la boutique en li...,Billing and Payments,Problem,medium,fr


In [68]:
classified_data.to_csv("classified_tickets_deepseek_500.csv", index=False)
print("Classification completed. Results saved to 'classified_tickets_deepseek_500.csv'.")

Classification completed. Results saved to 'classified_tickets_deepseek_500.csv'.
